# Synthetic Credit Card Fraud Scorecard

This notebook uses fake transactions to demonstrate transaction-gap and amount-spike features, a simple logistic regression, and ROC-AUC, Gini, and KS. All data are synthetic. A flag is only a review prompt, not proof of fraud.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, classification_report

rng = np.random.default_rng(42)
rows = []
start = pd.Timestamp('2026-01-01')
for account_num in range(150):
    account_id = f'ACCT-{account_num:04d}'
    typical_amount = rng.uniform(25, 250)
    current_time = start + pd.Timedelta(hours=int(rng.integers(0, 24)))
    for txn_num in range(45):
        label = int(rng.random() < 0.04)
        if label:
            gap_minutes = float(rng.uniform(1, 5))
            amount = typical_amount * float(rng.uniform(4, 8))
        else:
            gap_minutes = float(rng.exponential(8 * 60) + 10)
            amount = float(rng.lognormal(np.log(typical_amount), 0.45))
        current_time += pd.Timedelta(minutes=gap_minutes)
        rows.append({'account_id': account_id, 'transaction_time': current_time, 'amount': round(amount, 2), 'is_fraud': label})
transactions = pd.DataFrame(rows).sort_values(['account_id', 'transaction_time']).reset_index(drop=True)
print(f'{len(transactions):,} synthetic transactions; positive label rate: {transactions.is_fraud.mean():.2%}')
transactions.head()

## Features and transparent review rule
The amount baseline uses prior transactions only; the current transaction is excluded.

In [ ]:
grouped = transactions.groupby('account_id', sort=False)
transactions['seconds_since_previous'] = grouped['transaction_time'].diff().dt.total_seconds()
transactions['prior_30_avg_amount'] = grouped['amount'].transform(lambda s: s.shift(1).rolling(30, min_periods=5).mean())
transactions['amount_to_baseline_ratio'] = transactions['amount'] / transactions['prior_30_avg_amount']
transactions = transactions.replace([np.inf, -np.inf], np.nan)
transactions['rule_flag'] = ((transactions['seconds_since_previous'] <= 600) & (transactions['amount_to_baseline_ratio'] >= 3)).astype(int)
data = transactions.dropna(subset=['seconds_since_previous', 'amount_to_baseline_ratio']).copy()
print('Rule flag rate:', f'{data.rule_flag.mean():.2%}')
data[['seconds_since_previous', 'amount_to_baseline_ratio', 'rule_flag', 'is_fraud']].head()

## Logistic regression and evaluation
Gini = 2 × ROC-AUC − 1. KS is the maximum difference between true-positive and false-positive rates across thresholds.

In [ ]:
features = ['seconds_since_previous', 'amount_to_baseline_ratio']
X = data[features]
y = data['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
model.fit(X_train, y_train)
scores = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, scores)
fpr, tpr, thresholds = roc_curve(y_test, scores)
gini = 2 * auc - 1
ks = np.max(tpr - fpr)
print(f'Test ROC-AUC: {auc:.3f}')
print(f'Test Gini: {gini:.3f}')
print(f'Test KS: {ks:.3f}')
print('These are results on generated data, not evidence of real-world performance.')
print(classification_report(y_test, (scores >= 0.5).astype(int), zero_division=0))
plt.plot(fpr, tpr, label=f'ROC-AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve — synthetic test data')
plt.legend()
plt.show()

## Limitations
- Synthetic data and labels are intentionally patterned; good metrics do not establish real-world usefulness.
- Never upload real cardholder or confidential information to this public repository.
- Unusual activity can be legitimate. Flags should support human review, never automatic customer decisions.
- SQL feature examples are in `sql/feature_queries.sql`.